# FlexCNN Medical Physics - Notebook Template

This notebook demonstrates how to use the modular parameter system for FlexCNN.

**Key Features:**
- Import default parameters from `user_params.py`
- Override specific parameters without modifying source files
- Rebuild configurations after parameter changes
- Reload package code without kernel restart using `reload_package()`

## Setup Environment

In [16]:
import os
import sys
import shutil

def _detect_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

IN_COLAB = _detect_colab()

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    project_colab_dirPath = "/content/Colab/Working"
    drive_repo_dir = "/content/drive/MyDrive/Colab/Working"
    local_scripts_dir = os.path.join(project_colab_dirPath, "setup_functions")
    os.makedirs(local_scripts_dir, exist_ok=True)

    for filename in ("build_dicts.py", "script_setup.py", "user_params.py"):
        src = os.path.join(drive_repo_dir, "setup_functions", filename)
        dst = os.path.join(local_scripts_dir, filename)
        if not os.path.isfile(src):
            raise FileNotFoundError(f"Missing file on Drive: {src}")
        shutil.copy2(src, dst)

    if local_scripts_dir not in sys.path:
        sys.path.insert(0, local_scripts_dir)
else:
    scripts_dir = os.path.join(os.getcwd(), "setup_functions")
    if scripts_dir not in sys.path:
        sys.path.insert(0, scripts_dir)
    project_colab_dirPath = None

from setup_functions.script_setup import (
    sense_colab, sense_device, install_packages,
    setup_colab_environment, setup_local_environment,
    reload_package, refresh_repo
)

IN_COLAB = sense_colab()
print(f"Running in {'Colab' if IN_COLAB else 'Local'} environment")

Running in Local environment


In [4]:
# Setup environment (run once per session)
install_packages(IN_COLAB, ray_version=None)

if IN_COLAB:
    setup_colab_environment(
        github_username='petercl8',
        repo_name='FlexCNN_for_Medical_Physics',
        skip_git_update=False,
        force_fresh_clone=False  # Set to True if you get import errors after updating GitHub
    )
else:
    # Local setup - use 'walk' mode for fast reload
    setup_local_environment(
        repo_name='FlexCNN_for_Medical_Physics',
        mode='walk',
    )
# Test resources (function now available in global namespace)
list_compute_resources()

🖥️  Local environment detected. Installing CUDA-enabled PyTorch...
📦 Installing PyTorch with CUDA (cu124)...
✅ PyTorch installation complete.
📦 Installing other packages...
✅ Other packages installation complete.

CUDA Diagnostic Information:
✅ PyTorch version: 2.6.0+cu124
✅ CUDA available: True
✅ CUDA device count: 1
   GPU 0: NVIDIA GeForce RTX 3080 Ti Laptop GPU

📂 Added C:\Users\Peter Lindstrom\Desktop\FlexCNN_for_Medical_Physics to sys.path
📦 Loading FlexCNN_for_Medical_Physics package (walk mode)...


C:\Users\Peter Lindstrom\AppData\Roaming\jupyterlab-desktop\jlab_server\Lib\site-packages\hyperopt\atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


✨ Injecting all symbols into global namespace...
✅ Setup complete: 246 symbols loaded into globals.
CPUs available: 20
GPUs available: 1
  GPU 0: NVIDIA GeForce RTX 3080 Ti Laptop GPU


## Load Default Parameters

In [5]:
# Import after environment setup (build_dicts depends on package being importable)
from setup_functions.build_dicts import build_all_dicts
from user_params import get_params

# Get default parameters
params = get_params()

# Override Colab project directory from notebook-defined path
if IN_COLAB and project_colab_dirPath:
    params['project_colab_dirPath'] = project_colab_dirPath

# Configure plotting (function available from environment setup)
configure_plotting(plot_mode=params['plot_mode'])

# Display some key parameters
print(f"Run mode: {params['run_mode']}")
print(f"Network type: {params['network_type']}")
print(f"Train SI: {params['train_SI']}")
print(f"Gen sino size: {params['gen_sino_size']}")
print(f"Gen image size: {params['gen_image_size']}")

[INFO] Plot mode: inline - using interactive backend: module://matplotlib_inline.backend_inline
Run mode: train
Network type: ACT
Train SI: True
Gen sino size: 288
Gen image size: 180


## Override Parameters (Optional)

Modify parameters as needed without changing source files:

In [17]:
# Example: Change run mode and network type
#params['run_mode'] = 'train'
#params['network_type'] = 'ACT'
#params['train_checkpoint_file'] = 'my-custom-checkpoint'

print(f"Updated run mode: {params['run_mode']}")
print(f"Updated network type: {params['network_type']}")
print(f"Updated epochs: {params['train_epochs']}")

Updated run mode: train
Updated network type: ACT
Updated epochs: 100


## Setup Paths and Device

In [18]:
# Set main project directory (function available from environment setup)
project_dirPath = params['project_dirPath'] = setup_project_dirs(
    IN_COLAB,
    params['project_local_dirPath'],
    params['project_colab_dirPath'],
    mount_colab_drive=True
)

# Set device
device = params['device_opt'] = sense_device(device=params['device_opt'])

print(f"Project directory: {project_dirPath}")
print(f"Device: {device}")

Project directory: C:\Users\Peter Lindstrom\My Drive (lindstrom.peter@gmail.com)\Colab\Working
Device: cuda


## Build Configuration Dictionaries

In [19]:
# Build all dictionaries
all_dicts = build_all_dicts(params)

# Extract main dictionaries
config = all_dicts['config']
paths = all_dicts['paths']
settings = all_dicts['settings']
base_dirs = all_dicts['base_dirs']
tune_opts = all_dicts['tune_opts']
test_opts = all_dicts['test_opts']

print("✅ Configuration dictionaries built successfully!")
print(f"Config keys: {list(config.keys())}")

ValueError: 
❌ Network dimension or type mismatch detected!

The loaded configuration has different dimensions than your network_opts settings.
This usually means the checkpoint was trained with different input/output sizes.

Mismatches found:
  • gen_sino_size: config has 180, but network_opts specifies 288

Please update your network_opts to match the trained network, or use a different checkpoint.

# Visualizations

## 288x288 Networks

In [15]:
act_sino_array_name = 'train-highCountSino-320x257.npy'
act_image_array_name ='train-actMap.npy'
recon1_array = 'train-highCountImage.npy'
indexes = [100, 200, 300]
checkpoint_name = 'checkpoint-ACT-288-padZeros-tunedSSIM-100epochs'
network_type = 'ACT'
fig_size = 1

config_ACT_SI = { # 288x288, tuned SSIM, pad_type='zeros', interemediate size = None, horiz_pool=1, vert_pool=1
  "SI_alpha_min": -1,
  "SI_dropout": False,
  "SI_exp_kernel": 3,
  "SI_fixedScale": 1,
  "SI_gen_fill": 0,
  "SI_gen_final_activ": "LeakyReLU",
  "SI_gen_hidden_dim": 10,
  "SI_gen_mult": 1.5012782419950113,
  "SI_gen_neck": "narrow",
  "SI_gen_z_dim": 872,
  "SI_half_life_examples": -1,
  "SI_layer_norm": "instance",
  "SI_learnedScale_init": 7.305980552864529,
  "SI_normalize": False,
  "SI_output_scale_lr_mult": 1.6943444827125673,
  "SI_pad_mode": "zeros",
  "SI_skip_mode": "conv",
  "SI_stats_criterion": -1,
  "batch_base2_exponent": 7,
  "gen_b1": 0.3600790033157822,
  "gen_b2": 0.6033159868492163,
  "gen_image_channels": 1,
  "gen_image_size": 180,
  "gen_lr": 0.0024018267054557695,
  "gen_sino_channels": 3,
  "gen_sino_size": 288, # 288
  "network_type": "ACT",
  "sup_base_criterion": "MSELoss",
  "train_SI": True
}


paths = all_dicts['paths']


PlotPhantomRecons(indexes, checkpoint_name, network_type, 
                  config, paths, fig_size, device, settings,
                  outputs_to_plot = None,
                  act_image_array_name = act_image_array_name,
                  act_sino_array_name = act_sino_array_name, 
                  atten_image_array_name = None,
                  atten_sino_array_name = None, 
                  recon1_array_name = recon1_array, 
                  recon2_array_name = None)

'''
PlotPhantomRecons(indexes, checkpoint_name, network_type, 
                      config, paths, fig_size, device, settings, outputs_to_plot=None,
                      act_image_array_name, act_sino_array_name, atten_image_array_name=None,
                      atten_sino_array_name=None, recon1_array_name=recon1_array, recon2_array_name=None)
'''


NameError: name 'all_dicts' is not defined

# Reload Package After Code Changes

If you modify package code, use this cell to reload without restarting the kernel:

In [54]:
# Reload package after making code changes
reload_package()  # Reloads from local repo

# Or use refresh_repo() to pull from git and reinstall
# refresh_repo(IN_COLAB=IN_COLAB, force_fresh_clone=False)

# Rebuild dictionaries with updated code
all_dicts = build_all_dicts(params)
config = all_dicts['config']
paths = all_dicts['paths']
settings = all_dicts['settings']

print("✅ Package reloaded and dictionaries rebuilt!")

🔄 Reloading FlexCNN_for_Medical_Physics package...
✨ Injecting all symbols into global namespace...
✅ Reload complete: 246 symbols updated.
✅ Package reloaded and dictionaries rebuilt!


# Example: Quick Parameter Changes

Example of quickly changing parameters and re-running:

In [ ]:
# Quick experiment: visualize data
params['run_mode'] = 'visualize'
params['visualize_batch_size'] = 10
params['visualize_shuffle'] = True

# Rebuild and run
all_dicts = build_all_dicts(params)
run_pipeline(
    config=all_dicts['config'],
    paths=all_dicts['paths'],
    settings=all_dicts['settings'],
    tune_opts=all_dicts['tune_opts'],
    base_dirs=all_dicts['base_dirs'],
    test_opts=all_dicts['test_opts'],
)

# Run Pipeline

In [ ]:
# Run the pipeline (function available from environment setup)
run_pipeline(
    config=config,
    paths=paths,
    settings=settings,
    tune_opts=tune_opts,
    base_dirs=base_dirs,
    test_opts=test_opts,
)